NOTEBOOK INGESTION_BRONZE

Responsável por mover os dados brutos para a camada bronze, no formato delta e com colunas de auditoria. 

# CONFIGURAÇÕES GERAIS

## Rodar notebooks de configuração

In [0]:
%run ../../config/feat_squad2_config_adls

In [0]:
%run ../../utils/feat_squad2_utils

## Configurar as variáveis

In [0]:
entity_name = dbutils.widgets.get("entity_name")
folder_name_source = "real-time-data"
file_name_contains = f"ecommerce_{entity_name}.parquet"
path_target = f"abfs://{container_name_data_lake}/bronze/ecommerce_{entity_name}"
control_file_path = f"control/bronze/ecommerce_{entity_name}/control_file.json"
check_interval = 10
snapshot_id = 0

storage_options = {
    "storage_account_name": storage_account_name,
    "tenant_id": tenant_id,
    "client_id": client_id,
    "client_secret": client_secret
}

# EXECUÇÃO DO POLLING

In [0]:
while True:
    snapshot_id += 1

    log.info(f"Iniciando ciclo {snapshot_id} para entidade {entity_name}. Verificando novos arquivos")

    log.info("Lendo arquivos da raw") 

    all_files = list_files(
                        container_client=container_client_raw,
                        folder_name=folder_name_source,
                        file_name_contains=file_name_contains
    )

    log.info("Arquivos lidos com sucesso")

    processed_files = load_processed_files(
        container_client=container_client_data_lake,
        control_file_path=control_file_path
    )

    # Filtrar os arquivos que não foram processados

    list_files_to_ingest = [
        file_path
        for file_path in all_files
        if file_path not in processed_files
    ]
    
    # Variáveis de controle para métricas

    num_files = len(list_files_to_ingest)
    count_files_processed = 0
    total_records_processed = 0

    # Encerra o ciclo caso não tenha arquivos novos para processar

    if num_files == 0:
        log.info(
            f"Não há novos arquivos para ingerir. Encerrando ciclo {snapshot_id}."
        )

        time.sleep(check_interval)
        continue

    log.info(f"{num_files} novos arquivos encontrados. Iniciando ingestão.")

    # Loop de processamento de cada arquivo

    for file_path in list_files_to_ingest:
        count_files_processed +=1
        try:
            log.info(f"Processando arquivo {count_files_processed}/{num_files}: {file_path}")

            # Ler o arquivo atual

            df_snapshot = read_parquet_to_spark_df(
                container_client = container_client_raw,
                file_path = file_path
            )
            
            # Adicionar colunas de auditoria

            df_snapshot = (
                df_snapshot
                .withColumn("bronze_ingested_at", current_timestamp())
                .withColumn("bronze_source_file", lit(file_path))
            )

            df_snapshot_count = df_snapshot.count()
            total_records_processed += df_snapshot_count

            # Escrever no Data Lake

            write_adls(
                df=df_snapshot,
                path=path_target,
                storage_options=storage_options,
                mode="append"
            )
            
            # Incluir o arquivo processado no controle

            save_processed_file(
                container_client=container_client_data_lake,
                control_file_path=control_file_path,
                file_path=file_path
            )

            log.info(f"Arquivo {file_path} ingerido com sucesso. {df_snapshot_count} linhas processadas")

        except Exception as e:
            
            # Captura o erro em caso de falha no processamento do arquivo

            log.error(f"Erro no arquivo {file_path}. Mensagem: {str(e)}")

    log.info(f"Ciclo {snapshot_id} finalizado. {total_records_processed} linhas processadas no ciclo")

    # Aguarda a próxima janela de processamento
    
    time.sleep(check_interval)

TESTE

In [0]:
# Ler a tabela da Bronze
from deltalake import DeltaTable

dt = DeltaTable(
    f"abfs://squad2/bronze/ecommerce_{entity_name}",
    storage_options={
        "account_name": storage_options["storage_account_name"],
        "tenant_id": storage_options["tenant_id"],
        "client_id": storage_options["client_id"],
        "client_secret": storage_options["client_secret"]
    }
)

pdf = dt.to_pandas()

df_bronze = spark.createDataFrame(pdf)

display(df_bronze)

In [0]:
# Ler arquivo de controle
file_client = container_client_data_lake.get_file_client(control_file_path)

content = file_client.download_file().readall()

print(content.decode("utf-8"))

In [0]:
# Ler arquivos da fonte
all_files = list_files(
                    container_client = container_client_raw,
                    folder_name=folder_name_source,
                    file_name_contains=file_name_contains
)                    
print(f"print files in container:")
for files in all_files:
    print(files)

In [0]:
#Ler arquivos do destino
folder_name_target = f"bronze/ecommerce_{entity_name}"
all_files = list_files(
                    container_client = container_client_data_lake,
                    folder_name=folder_name_target,
)                    
print(f"print files in container:")
for files in all_files:
    print(files)

In [0]:
# Limpar arquivo de controle
file_client = container_client_data_lake.get_file_client(control_file_path)

file_client.upload_data(
    "[]",
    overwrite=True
)

print("Arquivo de controle limpo.")

In [0]:
# Deletar controle da bronze
container_client_data_lake.delete_directory(control_file_path)

In [0]:
# Deletar dados da bronze
container_client_data_lake.delete_directory(f"bronze/ecommerce_{entity_name}")